# Pneumonia Detection using MobileNetV2 (TensorFlow)

## Approach (inspired by high-scoring Kaggle notebooks)
- **Model**: MobileNetV2 (pretrained on ImageNet) with transfer learning
- **Strategy**: Freeze base → train classifier head → unfreeze top layers → fine-tune
- **Class Imbalance**: Handle with class weights + data augmentation
- **Augmentation**: Random horizontal flip, rotation, brightness/contrast
- **Metrics**: Accuracy, Precision, Recall, F1, Confusion Matrix, ROC-AUC

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models, callbacks
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from sklearn.metrics import (
    classification_report, confusion_matrix,
    roc_auc_score, precision_recall_fscore_support
)

print(f'TensorFlow: {tf.__version__}')
print(f'GPU: {tf.config.list_physical_devices("GPU")}')

In [ ]:
# Auto-detect dataset path
import glob

kaggle_input = '/kaggle/input'
possible_paths = glob.glob(os.path.join(kaggle_input, '**/chest_xray'), recursive=True)
if possible_paths:
    DATA_DIR = possible_paths[0]
else:
    for root, dirs, files in os.walk(kaggle_input):
        if 'NORMAL' in dirs and 'PNEUMONIA' in dirs:
            DATA_DIR = os.path.dirname(root)
            break

print(f'Dataset found at: {DATA_DIR}')
print(f'Contents: {os.listdir(DATA_DIR)}')

TRAIN_DIR = os.path.join(DATA_DIR, 'train')
VAL_DIR = os.path.join(DATA_DIR, 'val')
TEST_DIR = os.path.join(DATA_DIR, 'test')

# Hyperparameters
IMG_SIZE = 224
BATCH_SIZE = 32
NUM_EPOCHS = 25
NUM_CLASSES = 2
CLASS_NAMES = ['NORMAL', 'PNEUMONIA']

In [ ]:
# Data generators with augmentation
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=10,
    width_shift_range=0.05,
    height_shift_range=0.05,
    horizontal_flip=True,
    brightness_range=[0.8, 1.2],
    validation_split=0.0
)

val_test_datagen = ImageDataGenerator(rescale=1./255)

train_generator = train_datagen.flow_from_directory(
    TRAIN_DIR,
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='binary',
    classes=CLASS_NAMES,
    shuffle=True
)

val_generator = val_test_datagen.flow_from_directory(
    VAL_DIR,
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='binary',
    classes=CLASS_NAMES,
    shuffle=False
)

test_generator = val_test_datagen.flow_from_directory(
    TEST_DIR,
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='binary',
    classes=CLASS_NAMES,
    shuffle=False
)

print(f'Train: {train_generator.samples} | Val: {val_generator.samples} | Test: {test_generator.samples}')
print(f'Class indices: {train_generator.class_indices}')

In [ ]:
# Visualize sample images
fig, axes = plt.subplots(2, 5, figsize=(15, 6))
sample_imgs, sample_labels = next(train_generator)
for i in range(10):
    ax = axes[i // 5, i % 5]
    ax.imshow(sample_imgs[i])
    label = CLASS_NAMES[int(sample_labels[i])]
    ax.set_title(label, color='green' if label == 'PNEUMONIA' else 'blue')
    ax.axis('off')
plt.suptitle('Sample Training Images', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# Build MobileNetV2 model
base_model = MobileNetV2(
    input_shape=(IMG_SIZE, IMG_SIZE, 3),
    include_top=False,
    weights='imagenet'
)

# Freeze base model
base_model.trainable = False

# Build full model
model = models.Sequential([
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dropout(0.5),
    layers.Dense(256, activation='relu'),
    layers.Dropout(0.3),
    layers.Dense(1, activation='sigmoid')
])

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-4),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

model.summary()

In [ ]:
# Compute class weights for imbalanced dataset
from sklearn.utils.class_weight import compute_class_weight

class_weights_array = compute_class_weight(
    'balanced',
    classes=np.unique(train_generator.classes),
    y=train_generator.classes
)
class_weights = dict(enumerate(class_weights_array))
print(f'Class weights: {class_weights}')

In [ ]:
# Callbacks
cb_list = [
    callbacks.EarlyStopping(
        monitor='val_accuracy',
        patience=5,
        restore_best_weights=True,
        verbose=1
    ),
    callbacks.ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=2,
        min_lr=1e-7,
        verbose=1
    ),
    callbacks.ModelCheckpoint(
        'best_model.keras',
        monitor='val_accuracy',
        save_best_only=True,
        verbose=1
    )
]

In [ ]:
# Phase 1: Train classifier head only (base frozen)
print('='*60)
print('Phase 1: Training classifier head (base frozen)')
print('='*60)

history_phase1 = model.fit(
    train_generator,
    epochs=10,
    validation_data=val_generator,
    class_weight=class_weights,
    callbacks=cb_list
)

print(f'\nPhase 1 Best Val Acc: {max(history_phase1.history["val_accuracy"]):.4f}')

In [ ]:
# Phase 2: Unfreeze top layers and fine-tune
print('='*60)
print('Phase 2: Fine-tuning top layers')
print('='*60)

# Unfreeze base model
base_model.trainable = True

# Freeze all layers except last 30
for layer in base_model.layers[:-30]:
    layer.trainable = False

# Recompile with lower learning rate
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-5),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

print(f'Trainable layers: {sum(1 for l in model.layers if l.trainable)}')
print(f'Total parameters: {model.count_params():,}')
print(f'Trainable parameters: {sum(tf.size(v).numpy() for v in model.trainable_variables):,}')

history_phase2 = model.fit(
    train_generator,
    epochs=15,
    validation_data=val_generator,
    class_weight=class_weights,
    callbacks=cb_list
)

print(f'\nPhase 2 Best Val Acc: {max(history_phase2.history["val_accuracy"]):.4f}')

In [ ]:
# Plot training history
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

all_train_loss = history_phase1.history['loss'] + history_phase2.history['loss']
all_val_loss = history_phase1.history['val_loss'] + history_phase2.history['val_loss']
all_train_acc = history_phase1.history['accuracy'] + history_phase2.history['accuracy']
all_val_acc = history_phase1.history['val_accuracy'] + history_phase2.history['val_accuracy']

ax1.plot(all_train_loss, label='Train Loss')
ax1.plot(all_val_loss, label='Val Loss')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss')
ax1.set_title('Loss Over Epochs')
ax1.legend()
ax1.grid(True)

ax2.plot(all_train_acc, label='Train Acc')
ax2.plot(all_val_acc, label='Val Acc')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Accuracy')
ax2.set_title('Accuracy Over Epochs')
ax2.legend()
ax2.grid(True)

plt.tight_layout()
plt.savefig('training_history.png', dpi=150)
plt.show()

In [ ]:
# Evaluate on test set
test_generator.reset()
predictions = model.predict(test_generator)
y_pred = (predictions > 0.5).astype(int).flatten()
y_true = test_generator.classes
y_probs = predictions.flatten()

# Classification Report
print('Classification Report:')
print(classification_report(y_true, y_pred, target_names=CLASS_NAMES))

In [ ]:
# Confusion Matrix
cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES)
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Confusion Matrix')
plt.tight_layout()
plt.savefig('confusion_matrix.png', dpi=150)
plt.show()

# ROC-AUC
roc_auc = roc_auc_score(y_true, y_probs)
print(f'\nROC-AUC Score: {roc_auc:.4f}')

# Precision, Recall, F1
precision, recall, f1, _ = precision_recall_fscore_support(y_true, y_pred, average='binary')
print(f'Precision: {precision:.4f}')
print(f'Recall: {recall:.4f}')
print(f'F1-Score: {f1:.4f}')

In [ ]:
# Visualize predictions
test_generator.reset()
fig, axes = plt.subplots(2, 5, figsize=(18, 8))
imgs, labels = next(test_generator)

for i in range(10):
    ax = axes[i // 5, i % 5]
    ax.imshow(imgs[i])
    pred = CLASS_NAMES[y_pred[i]]
    true = CLASS_NAMES[int(labels[i])]
    color = 'green' if pred == true else 'red'
    ax.set_title(f'Pred: {pred}\nTrue: {true}', color=color, fontsize=10)
    ax.axis('off')

plt.suptitle('Test Predictions (Green=Correct, Red=Wrong)', fontsize=14)
plt.tight_layout()
plt.savefig('test_predictions.png', dpi=150)
plt.show()

In [ ]:
# Save model for Streamlit deployment
model.save('mobilenet_pneumonia.keras')

# Also save as .h5 for compatibility
model.save('mobilenet_pneumonia.h5')

print('Model saved as mobilenet_pneumonia.keras and .h5')
print(f'Final Test Accuracy: {(y_pred == y_true).mean():.4f}')
print(f'ROC-AUC: {roc_auc:.4f}')